In [1]:
import numpy as np

In [ ]:
def interpolate_signal(
    t_original: np.ndarray,
    x_original: np.ndarray,
    t_query: np.ndarray
) -> np.ndarray:
    """
    Interpolate using average of two neighboring samples.
    """
    return np.interp(t_query, t_original, x_original)

In [ ]:
def interpolate_signal_manual(
    t_original: np.ndarray,
    x_original: np.ndarray,
    t_query: np.ndarray
) -> np.ndarray:
    
    dt = t_original[1]-t_original[0]

    t_exact = (t_query - t_original[0])/dt

    max_idx = len(t_original)-1
    t_exact = np.clip(t_exact, 0 , max_idx)

    left = np.floor(t_exact).astype(int)
    right = np.clip(left+1, 0 , max_idx)

    x_left = x_original[left]
    x_right = x_original[right]

    is_close = np.isclose(t_exact, left)
    return np.where(is_close, x_left, 0.5*(x_left + x_right))



In [ ]:
def time_shift_signal(x, k):

    # roll shifts in circular way 
    x_shifted = np.roll(x,k)

    if k>0:
        x_shifted[:k] = 0
    else:
        x_shifted[k:] = 0


    # Using loop
    y = np.zeros_like(x)
    for i in range(len(x)):
        j = i + k
        if 0 <= j < len(x):
            y[i] = x[j]

    return x_shifted


def time_shift_sinusoid(n: np.ndarray, A: float, Omega0: float, phi: float, n0: int) -> np.ndarray:
    
    return A * np.cos(Omega0 * (n - n0) + phi)

In [ ]:
def time_compress_signal(x: np.ndarray, k: int) -> np.ndarray:
    y = np.zeros_like(x)
    
    # Range of time axis -8 to 8
    n = np.arange(-8, 9)
    
    valid_mask = (k * n >= -8) & (k * n <= 8)
    
    # Python index is always (time + 8)
    target_indices = n[valid_mask] + 8
    source_indices = (k * n[valid_mask]) + 8
    
    # Map the valid scaled values into the new array
    y[target_indices] = x[source_indices]
    
    return y

In [ ]:
def time_expand_signal(x: np.ndarray, k: int) -> np.ndarray:
    y = np.zeros_like(x)
    n = np.arange(-8, 9)
    
    valid_mask = (n % k == 0)
    
    target_indices = n[valid_mask] + 8
    source_indices = (n[valid_mask] // k) + 8
    
    y[target_indices] = x[source_indices]
    
    return y

In [ ]:
def time_scale_continuous(
    t: np.ndarray,
    x: np.ndarray,
    k: int
) -> np.ndarray:
    """
    Time sub-scaling:
        y(t) = x(t / k)
    """
    return interpolate_signal(t*k, t, x)

In [ ]:
def time_shift_sinusoid(n: np.ndarray, A: float, Omega0: float, phi: float, n0: int) -> np.ndarray:
    
    return A * np.cos(Omega0 * (n - n0) + phi)


def phase_change_sinusoid(n: np.ndarray, A: float, Omega0: float, phi: float, phi0: float) -> np.ndarray:
    
    return A * np.cos(Omega0 * n + (phi+phi0))


In [ ]:
def time_reversal(x):

    return x[::-1]

In [ ]:
def even_odd(x):

    x_reversed = time_reversal(x)
    even = 0.5 * (x + x_reversed)
    odd = 0.5 * (x - x_reversed)

    return even,odd

In [ ]:
def time_reverse_asymmetry(t, x):
    def sample(new_t):
     # x at index values, 0 if outside window
        v = np.zeros_like(new_t)
        ok = (new_t >= t[0]) & (new_t <= t[-1])
        v[ok] = x[new_t[ok] - t[0]]
        return v

    return sample(-t) 


In [ ]:
def transform(x, alpha, beta):
    x_result = time_shift_signal(x, beta)

    if alpha == 0:
        raise ValueError("Alpha cannot be zero for time scaling.")

    if alpha < 0:
        x_result = time_reversal(x_result)

    abs_alpha = abs(alpha)

    if 0 < abs_alpha < 1: 
        x_result = time_expand_signal(x_result, int(1 / abs_alpha))

    elif abs_alpha > 1:
        x_result = time_compress_signal(x_result, int(abs_alpha))
        
    return x_result